[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_28_DQN_FunctionApproximation.ipynb)

# Demo 28: From Q-Table to a Deep Q-Network

**挿入位置：DQN導入スライド**　｜　**実行環境：Google Colab**

連続的な2次元位置を持つ移動ロボットで、**Qテーブル（離散化）**と**DQN（ニューラルネットワーク）**を比較します。DQNにはExperience ReplayとTarget Networkを入れています。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
import torch
from torch import nn
from collections import deque

torch.set_num_threads(1)
class PointNav:
    def __init__(self,max_steps=40):self.max_steps=max_steps;self.goal=np.array([.86,.85])
    def reset(self,rng):
        self.pos=np.array([.10,.12])+rng.normal(0,.02,size=2);self.pos=np.clip(self.pos,0,1);self.t=0;return self.pos.copy()
    def step(self,a,rng):
        old=self.pos.copy();delta=np.array([[0,.12],[0,-.12],[-.12,0],[.12,0]])[a]
        self.pos=np.clip(old+delta+rng.normal(0,.015,2),0,1);self.t+=1
        dist_old=np.linalg.norm(self.goal-old);dist_new=np.linalg.norm(self.goal-self.pos)
        reached=dist_new<.12;done=reached or self.t>=self.max_steps
        # Shaped reward improves learnability in a short class demo; NOT sparse goal-only RL.
        reward=-.02 + .5*(dist_old-dist_new) + (2.0 if reached else 0.0)
        return self.pos.copy(),float(reward),done,reached

def encode(state,bins=8):return tuple(np.clip((state*bins).astype(int),0,bins-1))

def train_table(episodes=200,bins=8,seed=0):
    rng=np.random.default_rng(seed);env=PointNav();Q=np.zeros((bins,bins,4));scores=[];wins=[]
    for ep in range(episodes):
        s=env.reset(rng);total=0;success=False
        for t in range(env.max_steps):
            key=encode(s,bins);eps=max(.07,.5*(.985**ep))
            a=int(rng.integers(4)) if rng.random()<eps else int(rng.choice(np.flatnonzero(np.isclose(Q[key],Q[key].max()))))
            ns,r,done,reached=env.step(a,rng);nk=encode(ns,bins)
            Q[key+(a,)]+=.18*(r + (.97*np.max(Q[nk]) if not done else 0)-Q[key+(a,)])
            s=ns;total+=r;success=reached
            if done:break
        scores.append(total);wins.append(int(success))
    return Q,np.array(scores),np.array(wins)

class QNet(nn.Module):
    def __init__(self,width=48):
        super().__init__();self.net=nn.Sequential(nn.Linear(2,width),nn.ReLU(),nn.Linear(width,width),nn.ReLU(),nn.Linear(width,4))
    def forward(self,x):return self.net(x)

def train_dqn(episodes=200,width=48,seed=0):
    torch.manual_seed(seed);rng=np.random.default_rng(seed);env=PointNav()
    net=QNet(width);target=QNet(width);target.load_state_dict(net.state_dict())
    optim=torch.optim.Adam(net.parameters(),lr=.001);buffer=deque(maxlen=15000)
    scores=[];wins=[];updates=0
    for ep in range(episodes):
        s=env.reset(rng);total=0;success=False;eps=max(.05,.65*(.985**ep))
        for t in range(env.max_steps):
            with torch.no_grad():q=net(torch.tensor(s,dtype=torch.float32).unsqueeze(0))[0].numpy()
            a=int(rng.integers(4)) if rng.random()<eps else int(np.argmax(q))
            ns,r,done,reached=env.step(a,rng);buffer.append((s,a,r,ns,done));total+=r;success=reached
            s=ns
            if len(buffer)>=64 and (t%2==0):
                idx=rng.integers(len(buffer),size=64);sample=[buffer[int(i)] for i in idx]
                ss=torch.tensor(np.array([x[0] for x in sample]),dtype=torch.float32)
                aa=torch.tensor([x[1] for x in sample],dtype=torch.int64)
                rr=torch.tensor([x[2] for x in sample],dtype=torch.float32)
                nss=torch.tensor(np.array([x[3] for x in sample]),dtype=torch.float32)
                dd=torch.tensor([x[4] for x in sample],dtype=torch.float32)
                with torch.no_grad():y=rr+.97*(1-dd)*target(nss).max(1).values
                pred=net(ss).gather(1,aa[:,None]).squeeze(1)
                loss=nn.functional.smooth_l1_loss(pred,y)
                optim.zero_grad();loss.backward();nn.utils.clip_grad_norm_(net.parameters(),5.0);optim.step();updates+=1
                if updates%80==0:target.load_state_dict(net.state_dict())
            if done:break
        scores.append(total);wins.append(int(success))
    return net,np.array(scores),np.array(wins)

In [ ]:
epochs=widgets.IntSlider(value=180,min=60,max=500,step=20,description='Episodes',continuous_update=False)
bins_w=widgets.IntSlider(value=8,min=4,max=20,description='Q bins',continuous_update=False)
width_w=widgets.Dropdown(options=[16,32,48,64],value=48,description='NN width')
seed_w=widgets.IntSlider(value=3,min=0,max=100,description='Seed')
button=widgets.Button(description='Train table + DQN',button_style='primary');out=widgets.Output()
def do_train(_):
    with out:
        clear_output(wait=True);print('Training tabular Q-learning and DQN...')
        q,ret_tab,win_tab=train_table(epochs.value,bins_w.value,seed_w.value)
        net,ret_dqn,win_dqn=train_dqn(epochs.value,width_w.value,seed_w.value)
        clear_output(wait=True)
        fig,ax=plt.subplots(1,3,figsize=(14.5,3.7))
        k=min(25,epochs.value//2)
        for label,rr in [('Q-table',ret_tab),('DQN',ret_dqn)]:ax[0].plot(np.arange(k,len(rr)+1),np.convolve(rr,np.ones(k)/k,'valid'),label=label)
        ax[0].set(xlabel='Episode',ylabel='Return',title='Learning (smoothed)');ax[0].legend()
        for label,ww in [('Q-table',win_tab),('DQN',win_dqn)]:ax[1].plot(np.arange(k,len(ww)+1),np.convolve(ww,np.ones(k)/k,'valid'),label=label)
        ax[1].set(xlabel='Episode',ylabel='Success rate',title='Goal reached (smoothed)',ylim=(0,1));ax[1].legend()
        xx,yy=np.meshgrid(np.linspace(0,1,31),np.linspace(0,1,31))
        grid=torch.tensor(np.c_[xx.ravel(),yy.ravel()],dtype=torch.float32)
        with torch.no_grad():z=net(grid).max(1).values.numpy().reshape(xx.shape)
        im=ax[2].imshow(z,origin='lower',extent=[0,1,0,1],cmap='YlGnBu');ax[2].scatter([.86],[.85],c='red',marker='*',s=150);ax[2].set(title='DQN: max estimated Q(s,a)',xlabel='x',ylabel='y');fig.colorbar(im,ax=ax[2],shrink=.8)
        for a in ax[:2]:a.grid(alpha=.2)
        plt.tight_layout();plt.show()
        unseen=np.array([.37,.61]);print('New/unseen coordinate:',unseen,'| Q-table cell:',encode(unseen,bins_w.value),'| tabular Q:',np.round(q[encode(unseen,bins_w.value)],3))
        with torch.no_grad():print('NN estimates at same coordinate:',np.round(net(torch.tensor(unseen,dtype=torch.float32)).numpy(),3))
        print(f'Last-30 successes: table={win_tab[-30:].mean():.2f}, DQN={win_dqn[-30:].mean():.2f}')
button.on_click(do_train)
display(widgets.VBox([widgets.HBox([epochs,bins_w]),widgets.HBox([width_w,seed_w]),button,out]))
print('Click the train button. First run: typically a few seconds to a minute on Colab CPU.')

### ポイント
- Qテーブルは座標を有限個のマスへ丸めて保持します。ニューラルネットワークは座標を直接入力してQ値を推定します。
- DQNはミニバッチ再利用（Experience Replay）と、一定間隔で同期するTarget Networkを使います。
- **未経験状態への一般化は保証されません**。Q値は近似値なので、性能は学習量や乱数に依存します。
- 画像ベースDQNの仕組みを学ぶ前の軽量な2次元状態デモです（このネットワークはCNNではありません）。

## Try it yourself / 実験課題
- 座標の離散化幅を増減させ、テーブルの大きさと学習を比べる。
- 連続座標 (0.37, 0.61) に対する表とDQNの扱いの違いを説明する。
- Replay BufferとTarget Networkの役割を確認する。